## Attention HTML

In [1]:
import os
from pathlib import Path
import math
import io
from io import BytesIO
import base64

from PIL import Image
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import torch

from src.utils.read_write import read_yaml
from src.utils.data_utils import prepare_corn_data
from visualisation.att_visualisation_utils import overlay_attention, prepare_img, normalize_img, get_attentions, get_model

CWD = Path(os.getcwd()).parent
CONFIG_PATH = CWD / "configs" / "supervised_dino.yaml"
DATA_DIR = CWD / "data"
FILE_EXTENSION = ".tif"
DATA_SET = "CORN-3_orig"
DATA_SET_DIR = DATA_DIR / DATA_SET
DATA_SET_DIR

PosixPath('/Users/kim/PycharmProjects/cnf_ssl_clean/data/CORN-3_orig')

In [2]:
image_paths, _ = prepare_corn_data(DATA_SET_DIR, FILE_EXTENSION, sort=True)
PATCH_SIZE = 16
FINE_TUNED = False

In [3]:
MODEL_DIR = "2026-03-06_12-10-34_corn1500_dino_sl_clean"
FILE_NAME = "epoch=54-val/val_weighted_accuracy=0.85.ckpt"
MODEL_CHECKPOINT = Path(CWD) / "data" / "model_checkpoints" / MODEL_DIR / FILE_NAME

cfg = read_yaml(CONFIG_PATH)
model = get_model(cfg=cfg, fine_tuned=FINE_TUNED, patch_size=PATCH_SIZE, model_checkpoint=MODEL_CHECKPOINT)
device = torch.accelerator.current_accelerator()
for p in model.parameters():
    p.requires_grad = False
model.eval()
model.to(device)

=> loaded backbone from checkpoint 'dino/dino_vitbase16_pretrain/dino_vitbase16_pretrain_full_checkpoint.pth' with msg _IncompatibleKeys(missing_keys=[], unexpected_keys=['head.mlp.0.weight', 'head.mlp.0.bias', 'head.mlp.2.weight', 'head.mlp.2.bias', 'head.mlp.4.weight', 'head.mlp.4.bias', 'head.last_layer.weight'])


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
  (head): Identity()
)

In [4]:
def image_path_to_base64(image_path):
    img = Image.open(image_path).convert("L")

    # Convert image to base64
    buffered = BytesIO()
    img.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    return img_str

In [5]:
def attention_grid_to_base64(image, attn_tensor):
    """
    attn_tensor: (num_heads, H, W)
    """

    cmap = ListedColormap(["black", "red"])
    attn_np = attn_tensor.detach().cpu().numpy()

    num_heads = attn_np.shape[0]
    cols = 4
    rows = math.ceil(num_heads / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(10, 7))

    for i, ax in enumerate(axes.flat):
        if i < num_heads:
            masked_img = overlay_attention(image, attn_np[i])
            ax.imshow(masked_img, cmap=cmap)
            ax.set_title(f"H{i+1}", fontsize=8)
        ax.axis("off")

    plt.tight_layout()

    buffer = io.BytesIO()
    plt.savefig(buffer, format="png", dpi=150)
    plt.close(fig)

    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("utf-8")

In [12]:
def generate_html(image_paths, threshold=0.5, output_file="attention_overview.html"):

    html_content = """
    <html>
    <head>
        <title>Attention Overview</title>
        <style>
            body { font-family: Arial; }
            .row {
                display: flex;
                align-items: center;
                margin-bottom: 40px;
            }
            .column {
                margin-right: 30px;
            }
            img {
                border: 1px solid #ccc;
                max-width: 500px;
            }
        </style>
    </head>
    <body>
    <h1>Last Layer Attention Heads</h1>
    """

    for idx, img_path in enumerate(image_paths):

        img_name = img_path.parent.name + "_" + img_path.stem
        img_base64 = image_path_to_base64(img_path)

        img = Image.open(img_path)
        img = prepare_img(img, PATCH_SIZE).to(device)
        _, _, attn = get_attentions(model, img, threshold)
        attn_tensor = torch.from_numpy(attn)
        img_uint8 = normalize_img(img)
        image = img_uint8.squeeze().permute((1, 2, 0)).detach().cpu().numpy()
        attn_base64 = attention_grid_to_base64(image, attn_tensor)

        html_content += f"""
        <div class="row">
            <div class="column">
                <h3>Image {img_name}</h3>
                <img src="data:image/png;base64,{img_base64}" />
            </div>
            <div class="column">
                <h3>Attention Heads</h3>
                <img src="data:image/png;base64,{attn_base64}" />
            </div>
        </div>
        """

    html_content += "</body></html>"

    with open(output_file, "w") as f:
        f.write(html_content)

    print(f"Saved HTML to {output_file}")

In [13]:
generate_html(image_paths[:10], threshold=0.7)

Saved HTML to attention_overview.html
